# Embeddings and Semantic Search

## Notebook Contract
- **Objective:** demonstrate the v0.8 retrieval workflow: build an embedding index, run cosine-similarity search, and evaluate retrieval quality (Recall@k, MRR, nDCG@k).
- **Inputs:** a small synthetic FAQ-style corpus generated in the notebook (no Hub downloads).
- **Outputs:** sample query results, retrieval metric table, and a CSV summary under `reports/sample_run/`.
- **Default mode:** offline (TF-IDF embeddings). Flip `RUN_SENTENCE_TRANSFORMER=True` to compare against a real sentence-transformer model on the same corpus.
- **Expected runtime:** under 30 seconds offline; a few minutes on the sentence-transformer path depending on model size and network speed.
- **Scope boundary:** the index and metrics live in `src/hf_finetuning_lab/retrieval/`; the notebook orchestrates and visualizes.

## 1) Setup and Reproducibility

In [1]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path('..').resolve()
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from hf_finetuning_lab.retrieval import (
    EmbeddingIndex,
    IndexEntry,
    retrieval_report,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f"Timestamp (UTC): {datetime.now(UTC).isoformat(timespec='seconds')}")
print(f'Seed: {SEED}')

Python: 3.12.11
Platform: Windows-11-10.0.26200-SP0
Timestamp (UTC): 2026-05-18T18:46:25+00:00
Seed: 42


## 2) Parameters

`RUN_SENTENCE_TRANSFORMER=False` keeps the smoke run offline. Flipping it on triggers a model download from the Hub.

In [2]:
REPORTS_DIR = ROOT / 'reports' / 'sample_run'
RETRIEVAL_REPORT_PATH = REPORTS_DIR / 'retrieval_report.csv'
RETRIEVAL_SUMMARY_PATH = REPORTS_DIR / 'retrieval_summary.json'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 5
K_VALUES = (1, 3, 5)
SENTENCE_TRANSFORMER_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
RUN_SENTENCE_TRANSFORMER = False

print(f'Reports dir: {REPORTS_DIR}')
print(f'Top-k: {TOP_K} | k values for the report: {K_VALUES}')
print(f'Sentence-transformer path active: {RUN_SENTENCE_TRANSFORMER}')

Reports dir: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run
Top-k: 5 | k values for the report: (1, 3, 5)
Sentence-transformer path active: False


## 3) Synthetic FAQ Corpus and Queries

Each document has a stable `doc_id` and a short FAQ-style passage. The query set associates each natural-language question with the gold document the system should return at the top of the ranking.

In [3]:
@dataclass(slots=True, frozen=True)
class QueryItem:
    query: str
    relevant: tuple[str, ...]


documents: list[IndexEntry] = [
    IndexEntry(doc_id='d-account-reset', text='How to reset your password from the account settings page when you forget it.', metadata={'category': 'account'}),
    IndexEntry(doc_id='d-billing-refund', text='Request a refund for an incorrect charge through the billing support portal.', metadata={'category': 'billing'}),
    IndexEntry(doc_id='d-billing-invoice', text='Download and review your monthly invoice and adjust your billing email address.', metadata={'category': 'billing'}),
    IndexEntry(doc_id='d-tech-app-crash', text='Troubleshoot the mobile app when it crashes after opening the dashboard.', metadata={'category': 'technical'}),
    IndexEntry(doc_id='d-tech-export', text='Fix the export button when it does not respond in the desktop browser.', metadata={'category': 'technical'}),
    IndexEntry(doc_id='d-delivery-late', text='Track a late shipment and contact support when the parcel has not arrived.', metadata={'category': 'delivery'}),
    IndexEntry(doc_id='d-delivery-address', text='Change the delivery address on an order that has not yet shipped.', metadata={'category': 'delivery'}),
    IndexEntry(doc_id='d-cancel-plan', text='Cancel your subscription before the next renewal and confirm via email.', metadata={'category': 'cancellation'}),
    IndexEntry(doc_id='d-cancel-confirm', text='Receive a cancellation confirmation when your membership has ended.', metadata={'category': 'cancellation'}),
    IndexEntry(doc_id='d-security-locked', text='Recover an account that was locked after suspicious login activity.', metadata={'category': 'security'}),
    IndexEntry(doc_id='d-security-2fa', text='Enable two-factor authentication and review recent login attempts.', metadata={'category': 'security'}),
    IndexEntry(doc_id='d-account-email', text='Update the email address linked to your account from the profile menu.', metadata={'category': 'account'}),
]

queries: list[QueryItem] = [
    QueryItem('I forgot my password', ('d-account-reset',)),
    QueryItem('Got charged twice for my subscription', ('d-billing-refund',)),
    QueryItem('Mobile app keeps crashing on the dashboard', ('d-tech-app-crash',)),
    QueryItem('My parcel has not arrived yet', ('d-delivery-late',)),
    QueryItem('How do I change the delivery address?', ('d-delivery-address',)),
    QueryItem('Cancel my plan before renewal', ('d-cancel-plan',)),
    QueryItem('Suspicious login on my account', ('d-security-locked',)),
    QueryItem('Set up two-factor authentication', ('d-security-2fa',)),
    QueryItem('How do I download my invoice?', ('d-billing-invoice',)),
    QueryItem('Update the email on my account', ('d-account-email',)),
]

print(f'Corpus size: {len(documents)} | queries: {len(queries)}')
preview = pd.DataFrame(
    [
        {'doc_id': entry.doc_id, 'category': entry.metadata.get('category', ''), 'text': entry.text[:80]}
        for entry in documents[:5]
    ]
)
preview

Corpus size: 12 | queries: 10


,doc_id,category,text
0,d-account-reset,account,How to reset your password from the account se...
1,d-billing-refund,billing,Request a refund for an incorrect charge throu...
2,d-billing-invoice,billing,Download and review your monthly invoice and a...
3,d-tech-app-crash,technical,Troubleshoot the mobile app when it crashes af...
4,d-tech-export,technical,Fix the export button when it does not respond...


## 4) Offline Path: TF-IDF Embeddings

TF-IDF gives us deterministic, dense-enough embeddings without any model download. The index L2-normalizes rows so cosine similarity is a single matrix multiplication.

In [4]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=4000)
doc_matrix = vectorizer.fit_transform([entry.text for entry in documents]).toarray().astype(np.float32)
index = EmbeddingIndex(doc_matrix, documents)
print(f'Index size: {index.size} | dim: {index.dim}')

sample_query = queries[0].query
sample_vector = vectorizer.transform([sample_query]).toarray().astype(np.float32)[0]
results = index.search(sample_vector, k=TOP_K)
pd.DataFrame(
    [
        {
            'rank': rank,
            'doc_id': entry.doc_id,
            'score': round(score, 4),
            'text': entry.text,
        }
        for rank, (entry, score) in enumerate(results, start=1)
    ]
)

Index size: 12 | dim: 208


,rank,doc_id,score,text
0,1,d-account-reset,0.2069,How to reset your password from the account se...
1,2,d-security-2fa,0.0000,Enable two-factor authentication and review re...
2,3,d-cancel-plan,0.0000,Cancel your subscription before the next renew...
3,4,d-security-locked,0.0000,Recover an account that was locked after suspi...
4,5,d-cancel-confirm,0.0000,Receive a cancellation confirmation when your ...


## 5) Rank Every Query

Encode each query with the same vectorizer, search the index, and capture the ranked `doc_id` list per query.

In [5]:
def rank_queries(
    embeddings: np.ndarray,
    target_index: EmbeddingIndex,
    k: int,
) -> list[list[str]]:
    ranked: list[list[str]] = []
    for row in embeddings:
        hits = target_index.search(row, k=k)
        ranked.append([entry.doc_id for entry, _ in hits])
    return ranked


query_matrix = vectorizer.transform([item.query for item in queries]).toarray().astype(np.float32)
tfidf_rankings = rank_queries(query_matrix, index, k=TOP_K)
relevance = [item.relevant for item in queries]
ranking_preview = pd.DataFrame(
    {
        'query': [item.query for item in queries],
        'gold': [item.relevant[0] for item in queries],
        'top1': [r[0] if r else None for r in tfidf_rankings],
        'top3': [', '.join(r[:3]) for r in tfidf_rankings],
    }
)
ranking_preview

,query,gold,top1,top3
0,I forgot my password,d-account-reset,d-account-reset,"d-account-reset, d-security-2fa, d-cancel-plan"
1,Got charged twice for my subscription,d-billing-refund,d-cancel-plan,"d-cancel-plan, d-billing-refund, d-security-lo..."
2,Mobile app keeps crashing on the dashboard,d-tech-app-crash,d-tech-app-crash,"d-tech-app-crash, d-delivery-address, d-accoun..."
3,My parcel has not arrived yet,d-delivery-late,d-delivery-late,"d-delivery-late, d-delivery-address, d-cancel-..."
4,How do I change the delivery address?,d-delivery-address,d-delivery-address,"d-delivery-address, d-account-reset, d-account..."
5,Cancel my plan before renewal,d-cancel-plan,d-cancel-plan,"d-cancel-plan, d-security-locked, d-security-2fa"
6,Suspicious login on my account,d-security-locked,d-security-locked,"d-security-locked, d-delivery-address, d-secur..."
7,Set up two-factor authentication,d-security-2fa,d-security-2fa,"d-security-2fa, d-security-locked, d-cancel-plan"
8,How do I download my invoice?,d-billing-invoice,d-billing-invoice,"d-billing-invoice, d-account-reset, d-security..."
9,Update the email on my account,d-account-email,d-account-email,"d-account-email, d-delivery-address, d-cancel-..."


## 6) Retrieval Metrics

`retrieval_report` returns one row per requested `k` with Recall@k and nDCG@k, plus MRR (constant across rows).

In [6]:
tfidf_report = retrieval_report(tfidf_rankings, relevance, ks=K_VALUES)
tfidf_report.round(4)

,recall_at_k,ndcg_at_k,mrr
k,,,
1,0.9,0.9000,0.95
3,1.0,0.9631,0.95
5,1.0,0.9631,0.95


## 7) Optional: Sentence-Transformer Embeddings

Set `RUN_SENTENCE_TRANSFORMER=True` in the parameters cell to encode the same corpus with `sentence-transformers/all-MiniLM-L6-v2` and compare retrieval quality.

In [7]:
sbert_report = None
if RUN_SENTENCE_TRANSFORMER:
    from sentence_transformers import SentenceTransformer

    encoder = SentenceTransformer(SENTENCE_TRANSFORMER_MODEL)
    doc_emb = np.asarray(encoder.encode([entry.text for entry in documents], normalize_embeddings=False), dtype=np.float32)
    query_emb = np.asarray(encoder.encode([item.query for item in queries], normalize_embeddings=False), dtype=np.float32)
    sbert_index = EmbeddingIndex(doc_emb, documents)
    sbert_rankings = rank_queries(query_emb, sbert_index, k=TOP_K)
    sbert_report = retrieval_report(sbert_rankings, relevance, ks=K_VALUES)
    print(f'Sentence-transformer model: {SENTENCE_TRANSFORMER_MODEL}')
    display = sbert_report.round(4)
else:
    print('Sentence-transformer path skipped (set RUN_SENTENCE_TRANSFORMER=True to run).')
    display = pd.DataFrame()
display

Sentence-transformer path skipped (set RUN_SENTENCE_TRANSFORMER=True to run).


""


## 8) Side-by-Side Comparison and Persisted Summary

In [8]:
frames = {'tfidf': tfidf_report}
if sbert_report is not None:
    frames['sentence_transformer'] = sbert_report

long_report = (
    pd.concat(
        {name: frame for name, frame in frames.items()}, names=['method']
    )
    .reset_index()
    .rename(columns={'level_1': 'k'})
)
long_report.to_csv(RETRIEVAL_REPORT_PATH, index=False)
print(f'Report written to: {RETRIEVAL_REPORT_PATH}')
long_report.round(4)

Report written to: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\retrieval_report.csv


,method,k,recall_at_k,ndcg_at_k,mrr
0,tfidf,1,0.9,0.9000,0.95
1,tfidf,3,1.0,0.9631,0.95
2,tfidf,5,1.0,0.9631,0.95


In [9]:
summary = {
    'corpus_size': int(index.size),
    'query_count': int(len(queries)),
    'top_k': int(TOP_K),
    'tfidf_recall_at_1': float(tfidf_report.loc[1, 'recall_at_k']),
    'tfidf_recall_at_5': float(tfidf_report.loc[5, 'recall_at_k']),
    'tfidf_mrr': float(tfidf_report['mrr'].iloc[0]),
    'sentence_transformer_active': bool(RUN_SENTENCE_TRANSFORMER),
}
if sbert_report is not None:
    summary['sentence_transformer_recall_at_1'] = float(sbert_report.loc[1, 'recall_at_k'])
    summary['sentence_transformer_recall_at_5'] = float(sbert_report.loc[5, 'recall_at_k'])
    summary['sentence_transformer_mrr'] = float(sbert_report['mrr'].iloc[0])
RETRIEVAL_SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f'Summary written to: {RETRIEVAL_SUMMARY_PATH}')
summary

Summary written to: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\retrieval_summary.json


{'corpus_size': 12,
 'query_count': 10,
 'top_k': 5,
 'tfidf_recall_at_1': 0.9,
 'tfidf_recall_at_5': 1.0,
 'tfidf_mrr': 0.95,
 'sentence_transformer_active': False}

## 9) Error Inspection

List the queries where the TF-IDF index did not place the gold document at rank 1. These are the natural candidates for richer embeddings or re-ranking.

In [10]:
errors = []
for item, ranking in zip(queries, tfidf_rankings, strict=True):
    gold = item.relevant[0]
    rank = ranking.index(gold) + 1 if gold in ranking else None
    if rank != 1:
        errors.append(
            {
                'query': item.query,
                'gold': gold,
                'top1': ranking[0] if ranking else None,
                'gold_rank_in_top_k': rank,
            }
        )
pd.DataFrame(errors)

,query,gold,top1,gold_rank_in_top_k
0,Got charged twice for my subscription,d-billing-refund,d-cancel-plan,2


## 10) Checklist
- `EmbeddingIndex` is encoder-agnostic: hand it any L2-normalisable matrix and a list of `IndexEntry` records.
- TF-IDF is the default offline encoder so smoke runs without a model download; sentence-transformers slot in via the same index when `RUN_SENTENCE_TRANSFORMER=True`.
- `retrieval_report` returns Recall@k, nDCG@k, and MRR; pair it with the v0.4 bootstrap utility for confidence intervals on these aggregates.
- For production: replace the in-notebook corpus with a real document store, switch to an ANN index (e.g. FAISS or pgvector) past a few hundred documents, and add a cross-encoder re-ranker on the top-k candidates.